In [3]:
import pandas as pd
import numpy as np

In [4]:
hybrid_df = pd.read_parquet("/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/Embedding/data_outputs/13_candidate_job_hybrid_ranking.parquet")
print(hybrid_df.columns.tolist())
print(hybrid_df.shape)
hybrid_df.head(3)

['candidate_id', 'job_id', 'job_title', 'skill_overlap_score', 'group_similarity_score', 'dominant_group_score', 'baseline_score', 'match_explanation', 'baseline_score_norm', 'semantic_similarity', 'semantic_rank', 'semantic_score_norm', 'hybrid_taxonomy_score', 'hybrid_no_taxonomy_score', 'hybrid_score', 'rank_taxonomy', 'rank_no_taxonomy', 'hybrid_rank']
(100, 18)


,candidate_id,job_id,job_title,skill_overlap_score,group_similarity_score,dominant_group_score,baseline_score,match_explanation,baseline_score_norm,semantic_similarity,semantic_rank,semantic_score_norm,hybrid_taxonomy_score,hybrid_no_taxonomy_score,hybrid_score,rank_taxonomy,rank_no_taxonomy,hybrid_rank
0,C001,J001,Java Backend Developer,1.0000,1.0,1,1.0000,skill overlap=1.0; group similarity=1.0; same ...,1.0000,0.897339,1,0.948670,0.984601,0.984601,0.984601,1,1,1
1,C001,J025,Database Developer,0.3333,0.5,1,0.4417,skill overlap=0.3333; group similarity=0.5; sa...,0.4417,0.873763,3,0.936882,0.590254,0.514374,0.590254,2,2,2
2,C001,J027,.NET Backend Developer,0.2000,0.5,1,0.3550,skill overlap=0.2; group similarity=0.5; same ...,0.3550,0.867865,7,0.933932,0.528680,0.420180,0.528680,3,3,3


In [3]:
def score_level(score):
    if score >= 0.85:
        return "rất cao"
    elif score >= 0.70:
        return "cao"
    elif score >= 0.55:
        return "khá"
    else:
        return "trung bình"

In [4]:
def build_explanation_short(row):
    reasons = []

    if row.get("skill_overlap_score", 0) >= 0.6:
        reasons.append("trùng kỹ năng chính")
    if row.get("group_similarity_score", 0) >= 0.6:
        reasons.append("cùng nhóm nghề")
    if row.get("dominant_group_score", 0) >= 0.8:
        reasons.append("phù hợp nhóm chính")
    if row.get("semantic_similarity", 0) >= 0.8:
        reasons.append("semantic match cao")

    if not reasons:
        return "Mức độ phù hợp khá tốt với công việc"

    return "Phù hợp vì " + ", ".join(reasons[:3])

In [5]:
def build_explanation_long(row):
    hybrid = row.get("hybrid_score", 0)
    semantic = row.get("semantic_similarity", 0)
    skill = row.get("skill_overlap_score", 0)
    group_sim = row.get("group_similarity_score", 0)
    dominant = row.get("dominant_group_score", 0)

    parts = []
    parts.append(
        f"Công việc này có mức độ phù hợp {score_level(hybrid)} với ứng viên "
        f"(hybrid score = {hybrid:.3f})."
    )

    if skill >= 0.6:
        parts.append("Hồ sơ có mức trùng khớp tốt về kỹ năng đã chuẩn hóa.")
    elif skill >= 0.4:
        parts.append("Hồ sơ có mức trùng khớp kỹ năng ở mức khá.")
    else:
        parts.append("Mức trùng khớp kỹ năng trực tiếp chưa quá cao.")

    if group_sim >= 0.6 or dominant >= 0.8:
        parts.append("Ứng viên và công việc nằm gần nhau về nhóm nghề theo taxonomy.")
    
    if semantic >= 0.8:
        parts.append("Độ tương đồng ngữ nghĩa giữa hồ sơ và mô tả công việc ở mức cao.")
    elif semantic >= 0.65:
        parts.append("Độ tương đồng ngữ nghĩa ở mức khá tốt.")

    match_expl = str(row.get("match_explanation", "")).strip()
    if match_expl and match_expl.lower() != "nan":
        parts.append(f"Giải thích baseline: {match_expl}")

    return " ".join(parts)

In [6]:
def build_ui_badges(row):
    badges = []

    if row.get("hybrid_rank", 999) <= 3:
        badges.append("Top match")
    if row.get("skill_overlap_score", 0) >= 0.6:
        badges.append("Skill match")
    if row.get("group_similarity_score", 0) >= 0.6:
        badges.append("Same group")
    if row.get("semantic_similarity", 0) >= 0.8:
        badges.append("Semantic fit")

    return badges

In [7]:
explain_df = hybrid_df.copy()

explain_df["explanation_short"] = explain_df.apply(build_explanation_short, axis=1)
explain_df["explanation_long"] = explain_df.apply(build_explanation_long, axis=1)
explain_df["ui_badges"] = explain_df.apply(build_ui_badges, axis=1)

explain_df.head(5)

,candidate_id,job_id,job_title,skill_overlap_score,group_similarity_score,dominant_group_score,baseline_score,match_explanation,baseline_score_norm,semantic_similarity,semantic_rank,semantic_score_norm,hybrid_score,hybrid_rank,explanation_short,explanation_long,ui_badges
0,C001,J001,Backend Developer,1.0,1,1,1.0,skill overlap=1.0; group similarity=1.0; same ...,1.0,0.917700,1,0.958850,0.983540,1,"Phù hợp vì trùng kỹ năng chính, cùng nhóm nghề...",Công việc này có mức độ phù hợp rất cao với ứn...,"[Top match, Skill match, Same group, Semantic ..."
1,C001,J003,Fullstack Developer,0.0,1,1,0.4,group similarity=1.0; same dominant group,0.4,0.861202,2,0.930601,0.612240,2,"Phù hợp vì cùng nhóm nghề, phù hợp nhóm chính,...",Công việc này có mức độ phù hợp khá với ứng vi...,"[Top match, Same group, Semantic fit]"
2,C001,J002,Frontend Developer,0.0,1,1,0.4,group similarity=1.0; same dominant group,0.4,0.858543,3,0.929271,0.611709,3,"Phù hợp vì cùng nhóm nghề, phù hợp nhóm chính,...",Công việc này có mức độ phù hợp khá với ứng vi...,"[Top match, Same group, Semantic fit]"
3,C001,J008,Business Analyst,0.0,1,1,0.4,group similarity=1.0; same dominant group,0.4,0.838776,4,0.919388,0.607755,4,"Phù hợp vì cùng nhóm nghề, phù hợp nhóm chính,...",Công việc này có mức độ phù hợp khá với ứng vi...,"[Same group, Semantic fit]"
4,C001,J004,Data Analyst,0.0,1,1,0.4,group similarity=1.0; same dominant group,0.4,0.834230,6,0.917115,0.606846,5,"Phù hợp vì cùng nhóm nghề, phù hợp nhóm chính,...",Công việc này có mức độ phù hợp khá với ứng vi...,"[Same group, Semantic fit]"


In [8]:
ui_explain_df = explain_df[
    [
        "candidate_id",
        "job_id",
        "job_title",
        "hybrid_score",
        "hybrid_rank",
        "semantic_similarity",
        "skill_overlap_score",
        "group_similarity_score",
        "dominant_group_score",
        "explanation_short",
        "explanation_long",
        "ui_badges"
    ]
].copy()

ui_explain_df.head(10)

,candidate_id,job_id,job_title,hybrid_score,hybrid_rank,semantic_similarity,skill_overlap_score,group_similarity_score,dominant_group_score,explanation_short,explanation_long,ui_badges
0,C001,J001,Backend Developer,0.983540,1,0.917700,1.0,1,1,"Phù hợp vì trùng kỹ năng chính, cùng nhóm nghề...",Công việc này có mức độ phù hợp rất cao với ứn...,"[Top match, Skill match, Same group, Semantic ..."
1,C001,J003,Fullstack Developer,0.612240,2,0.861202,0.0,1,1,"Phù hợp vì cùng nhóm nghề, phù hợp nhóm chính,...",Công việc này có mức độ phù hợp khá với ứng vi...,"[Top match, Same group, Semantic fit]"
2,C001,J002,Frontend Developer,0.611709,3,0.858543,0.0,1,1,"Phù hợp vì cùng nhóm nghề, phù hợp nhóm chính,...",Công việc này có mức độ phù hợp khá với ứng vi...,"[Top match, Same group, Semantic fit]"
3,C001,J008,Business Analyst,0.607755,4,0.838776,0.0,1,1,"Phù hợp vì cùng nhóm nghề, phù hợp nhóm chính,...",Công việc này có mức độ phù hợp khá với ứng vi...,"[Same group, Semantic fit]"
4,C001,J004,Data Analyst,0.606846,5,0.834230,0.0,1,1,"Phù hợp vì cùng nhóm nghề, phù hợp nhóm chính,...",Công việc này có mức độ phù hợp khá với ứng vi...,"[Same group, Semantic fit]"
5,C002,J002,Frontend Developer,0.984192,1,0.920958,1.0,1,1,"Phù hợp vì trùng kỹ năng chính, cùng nhóm nghề...",Công việc này có mức độ phù hợp rất cao với ứn...,"[Top match, Skill match, Same group, Semantic ..."
6,C002,J001,Backend Developer,0.613547,2,0.867735,0.0,1,1,"Phù hợp vì cùng nhóm nghề, phù hợp nhóm chính,...",Công việc này có mức độ phù hợp khá với ứng vi...,"[Top match, Same group, Semantic fit]"
7,C002,J003,Fullstack Developer,0.612434,3,0.862168,0.0,1,1,"Phù hợp vì cùng nhóm nghề, phù hợp nhóm chính,...",Công việc này có mức độ phù hợp khá với ứng vi...,"[Top match, Same group, Semantic fit]"
8,C002,J008,Business Analyst,0.604694,4,0.823472,0.0,1,1,"Phù hợp vì cùng nhóm nghề, phù hợp nhóm chính,...",Công việc này có mức độ phù hợp khá với ứng vi...,"[Same group, Semantic fit]"
9,C002,J004,Data Analyst,0.603842,5,0.819211,0.0,1,1,"Phù hợp vì cùng nhóm nghề, phù hợp nhóm chính,...",Công việc này có mức độ phù hợp khá với ứng vi...,"[Same group, Semantic fit]"


In [9]:
ui_explain_df.to_parquet("/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/Embedding/data_outputs/14_candidate_job_explanations.parquet", index=False)
ui_explain_df.to_excel("/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/Embedding/data_outputs/14_candidate_job_explanations.xlsx", index=False)

print("Saved -> 14_candidate_job_explanations.parquet")
print("Saved -> 14_candidate_job_explanations.xlsx")

Saved -> 14_candidate_job_explanations.parquet
Saved -> 14_candidate_job_explanations.xlsx
